> **Restored after the `eye_movements.pkl` refactor.**
>
> The per-target `icon{i}_distance_dva` columns this notebook used to read no longer exist - they were
> unique per trial, so concatenating subjects produced a 97%-NaN table. Distances now come from
> `DataStore.fixation_target_dists` (built by `pipeline.stage2_align.fixations_to_icons`), which is
> already in the long format every estimator needs. See `CODE_REVIEW.md` T4.

# Functional Visual Field (FVF)

Determine each subject's **functional visual field** - the radius around fixation from which a target is
detected and selected for foveation. This is the input to an inspection-conditioned d' denominator
(`CODE_REVIEW.md` T1), which would replace "every non-target icon" with "items plausibly inspected".

<i>TLDR:</i> **use estimator C, the selection hazard: FVF = 4.34 DVA pooled** (per subject 4.11-5.53).
That is **2.5x** the independently-derived `ON_TARGET_THRESHOLD_DVA = 1.75`, which is the relationship a
valid field estimate should have - see the validation section. Estimators A and B were tried first and
**both fail on this data**; they are kept below because knowing *why* they fail is what justifies C.

---

## The three estimators

### Why not the obvious one
`P(identified | min eccentricity from any fixation)` is **circular**. Marking a target requires foveating
it, so an identification counts as a hit only when gaze is within `ON_TARGET_THRESHOLD_DVA`. Every hit
therefore has minimum eccentricity below that threshold *by construction*: the curve degenerates into a
step function at the on-target threshold and recovers nothing about peripheral detection.

### A - foveation falloff &nbsp;&nbsp; <span style='color:#b00'>(fails: saturated)</span>
`P(target ever foveated | closest approach that was NOT itself on-target)`, over the whole trial.
Outcome is foveation rather than identification, which removes the circularity and separates the
constructs: the curve's *level* near the fovea carries the LWS floor, its *falloff* the field size.

**Why it fails here:** the predictor has almost no spread - p50 2.04, p95 3.54, p99 4.93 DVA. With ~195
fixations per trial over a ~34 x 19 DVA array, essentially *every* target is approached closely at some
point, whether or not it was ever detected. `P(foveated)` descends only 0.96 -> 0.82 and never reaches half
its asymptote, so there is no falloff point to read. The estimator returns `NaN` rather than the edge of
the observed range, which would have looked like an answer.

### B - saccade-launch distance &nbsp;&nbsp; <span style='color:#b00'>(fails: wrong quantity)</span>
For each foveated target, the distance from the fixation immediately preceding its first on-target
fixation - the fixation the subject launched from. FVF = the 95th percentile.

**Why it fails here:** it returns ~13.4 DVA, on an array only ~19 DVA tall. In a real scanpath the
preceding fixation is usually just wherever the subject happened to be scanning, not a deliberate approach,
so B approximates the 95th percentile of **saccade amplitude** rather than a detection radius.

### C - selection hazard &nbsp;&nbsp; <span style='color:#070'>(works)</span>
For **every fixation** at which a target was still unfoveated: `P(the next saccade lands on that target |
current distance to it)`. A discrete-time hazard rather than a per-target outcome.

**Why it works:** it never aggregates over the trial, so it cannot saturate the way A does - most
opportunities at any distance are declined (base rate ~2.5%), leaving the curve room to fall. And it counts
only the fixation that actually preceded selection, so unlike B it is not diluted by incidental scanning.

All three recover a known radius from synthetic data (true 4.0 -> A 3.9, B 3.8, C 3.9), so the divergence
below is a property of the real scanpaths, not of the implementations.

In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import config as cnfg
from analysis.helpers.read_data import load_data
from analysis.fvf.fvf import (
    estimate_fvf, estimate_by_foveation_falloff, estimate_by_launch_distance,
    estimate_by_selection_hazard, selection_opportunities,
)

pio.renderers.default = 'notebook'      # 'notebook' or 'browser'

# compare against the a-priori constant rather than hard-coding a number
ON_TARGET = cnfg.ON_TARGET_THRESHOLD_DVA
PERCENTILES = [0.5, 0.75, 0.85, 0.9, 0.95, 0.99]

### Read data

In [ ]:
loaded_data = load_data(cnfg.OUTPUT_PATH)
dists = loaded_data.fixation_target_dists
idents = loaded_data.identifications
del loaded_data

## (A) Foveation falloff - demonstrating the saturation

The point of running A is to *see* the saturation: the approach-distance distribution is compressed into a
few DVA, and the foveation rate barely moves across it.

In [ ]:
falloff_by_subject, falloff_pooled, curve_a = estimate_by_foveation_falloff(dists, ON_TARGET)

print(f'Pooled FVF (A): {falloff_pooled}   <- NaN means the curve never fell to half its asymptote')
print(f"Near-fovea foveation rate: {curve_a['rate'].iloc[0]:.3f}"
      f"   lowest reached: {curve_a['rate'].min():.3f}")
print(f"Dynamic range: {curve_a['rate'].iloc[0] / max(curve_a['rate'].min(), 1e-9):.1f}x"
      f'   (C achieves ~40x - see below)')
curve_a

## (B) Saccade-launch distance - demonstrating the wrong quantity

Compare the result against the array size (~34 x 19 DVA) and against typical saccade amplitudes.

In [ ]:
launch_by_subject, launch_pooled, launches = estimate_by_launch_distance(dists, ON_TARGET)

print(f'Pooled FVF (B): {launch_pooled:.2f} DVA   (n = {len(launches):,} launches)')
print(f'That is {launch_pooled / ON_TARGET:.1f}x the on-target threshold, on an array ~19 DVA tall.')

launch_summary = (
    pd.concat([
        launches['launch_dva'].describe(PERCENTILES).rename('all'),
        launches.groupby('subject', observed=True)['launch_dva'].describe(PERCENTILES).T,
    ], axis=1)
).T
launch_summary

## Estimator C in detail

### What a hazard is

In event-history analysis a **hazard** is the probability that an event happens *now*, given that it has not
happened yet:

$$h(t) = P(\text{event at } t \mid \text{no event before } t)$$

The conditioning is the whole point. A marginal probability ("was this target ever foveated?") mixes together
everything that happened over the trial. A hazard asks a much narrower question at each moment, of only those
cases still **at risk** - here, targets not yet foveated.

C replaces time with **distance** as the conditioning variable:

$$h(d) = P(\text{the next saccade lands on this target} \mid \text{still unfoveated, currently } d \text{ away})$$

Read plainly: *the subject is fixating somewhere, a target sits `d` degrees away and has not been looked at
yet - how often does the very next eye movement go to it?* Where that probability collapses is the edge of the
region the subject is effectively drawing on.

### Why the hazard framing rescues the estimate

Estimator A asked one question per target: *was it ever foveated, given the closest the subject ever came?*
Both sides of that question are compressed by the data:

- the **predictor** saturates - with ~195 fixations per trial over a ~34 x 19 DVA array, essentially every
  target is approached closely at some point (p50 2.04, p95 3.54 DVA), detected or not;
- the **outcome** saturates too - 89.6% of targets are eventually foveated.

With little variance on either side there is nothing to fit, and the curve descends only 0.96 -> 0.82.

C changes the unit of analysis from *the target* to *the opportunity*. Every (fixation, unfoveated target) pair
is one observation: **109,011** of them, of which only **2,734 (2.5%)** end in selection. The base rate is low,
so the outcome has room to vary; and a fixation can be anywhere relative to a target, so the predictor spans
the full range. That is why C's curve falls by 40x where A's barely moves.

### Step by step

**1. Reshape to long form** (`per_target_distances`). The fixations table is wide - one `{icon}_distance_dva`
column per target. Melt it to one row per (fixation, target), and flag
`on_target = distance_dva <= ON_TARGET_THRESHOLD_DVA`.

**2. Build the risk set** (`selection_opportunities`). Within each (subject, trial, eye, target), sort
fixations by `event` (time order) and find `first`, the index of the first on-target fixation:

- every fixation at index `0 … first-1` is an **opportunity** - at that moment the target was unfoveated, so
  it was at risk of being selected;
- if the target *was* foveated, exactly one opportunity - index `first-1`, the fixation immediately before -
  is marked `selected`. That is the fixation the successful saccade launched from;
- if the target was **never** foveated, all its opportunities are unselected. In survival terms these are
  right-censored: the trial ended before the event. They belong in the denominator, and counting them is what
  keeps the hazard honest;
- if `first == 0` the target was already on-target at the first fixation, so there was never an opportunity to
  select it - the pair is dropped.

**3. Bin by distance.** Equal-count bins (`pd.qcut`), so every bin carries the same number of opportunities and
the noise is comparable across the curve. Per bin: `centre` = median distance, `rate` = mean of `selected`,
which is the empirical hazard at that distance.

**4. Read off the field edge** (`_falloff_point`). Take the hazard in the nearest bin as the near-fovea value
$h_0$, find the first bin where $h \le h_0/2$, and linearly interpolate between it and the previous bin. If
the curve never gets there, return `NaN` - **not** the last bin centre, which would look like an estimate
while being nothing but the edge of the data. That is exactly the trap estimator A falls into here.

**5. Repeat per subject**, requiring at least `n_bins x 10` opportunities so a subject with sparse data does
not contribute a curve fitted to noise.

### Why the estimate must exceed `ON_TARGET_THRESHOLD_DVA`

The two radii answer different questions:

| | question | value |
| --- | --- | --- |
| `ON_TARGET_THRESHOLD_DVA` | how close must gaze be to count as looking **at** the target? | 1.75 DVA |
| FVF | from how far away can a target be **noticed** and selected? | 4.34 DVA |

**The lower bound is a logical necessity.** If FVF were smaller than the on-target radius, a target could only
be noticed once gaze was already on it - and then search would be an unguided random walk, with each item found
only by landing on it by chance. Subjects find targets far faster than that predicts. The data says so
directly: the hazard at ~2.9 DVA is **0.204**, so one in five fixations sitting ~2.9 DVA from an unfoveated
target is immediately followed by a saccade to it. Selection is being driven from outside the on-target radius,
which *is* the evidence that FVF > 1.75.

**The upper bound is a plausibility check.** The array spans ~34 x 19 DVA. A field of 13 DVA - estimator B's
answer - would mean a subject can detect a target across most of the display, which would make search nearly
instantaneous and Looking-Without-Seeing impossible. So a credible estimate sits well inside the array:
`ON_TARGET < FVF << array extent`. C's 4.34 DVA is 2.5x the threshold and about a quarter of the array height,
and matches the 2-5 deg range typical for cluttered visual-search displays.

**Stability is the third check.** The on-target threshold is near-identical across subjects, so an estimator
tracking a real perceptual quantity should be too. C's per-subject SD is **0.39 DVA** against B's **1.83**.

### What C does and does not measure

Being precise about this matters for how the value is used. The hazard at distance `d` depends not only on
whether the target is perceptible from there, but also on how many **other** items are competing for the next
saccade - and there are geometrically more items at larger distances. So C estimates a *functional* field: the
distance beyond which a target stops winning the competition for the next fixation. It is **not** a pure
perceptual detection threshold, which would need a task with no competing items.

For the intended use - an inspection-conditioned d' denominator - the functional quantity is the right one:
the question there is which items were effectively inspected, not which were theoretically resolvable.

## (C) Selection hazard - the usable estimator

Every (fixation, not-yet-foveated target) pair is an opportunity to select that target; the fixation that
actually preceded foveation is the one that took it.

In [ ]:
opportunities = selection_opportunities(dists, ON_TARGET)
print(f'{len(opportunities):,} selection opportunities, '
      f"{int(opportunities['selected'].sum()):,} taken "
      f"(base rate {100 * opportunities['selected'].mean():.2f}%)")

hazard_by_subject, hazard_pooled, curve_c = estimate_by_selection_hazard(dists, ON_TARGET)
print(f'\nPooled FVF (C): {hazard_pooled:.2f} DVA')
print(f"Hazard falls {curve_c['rate'].iloc[0]:.4f} -> {curve_c['rate'].iloc[-1]:.4f} "
      f"({curve_c['rate'].iloc[0] / max(curve_c['rate'].iloc[-1], 1e-9):.0f}x) - a real half-point, not a censored one")
curve_c

In [ ]:
hazard_by_subject.describe(PERCENTILES).to_frame('selection_hazard_dva').T

## Validation against `ON_TARGET_THRESHOLD_DVA`

`ON_TARGET_THRESHOLD_DVA = 1.75` was derived independently, in
`pipeline/_determine_on_target_threshold.ipynb`, as the value that cleanly separates hits from false alarms
(it covers 100% of hits and sits just below the smallest false-alarm distance). It is therefore a trusted
yardstick, and a valid FVF estimate must stand in a specific relationship to it:

1. **Larger than it.** The region from which a target can be *detected* must exceed the radius within which
   gaze already counts as being *on* it - otherwise targets could only ever be found by landing on them by
   chance, and search would be impossible.
2. **Same order of magnitude.** A field several times the array height is not a field; it means the
   estimator is measuring something else.
3. **Consistent across subjects.** The on-target threshold is stable across subjects, so an estimator
   tracking a real perceptual quantity should be too.

The cell below scores all three estimators on these criteria.

In [ ]:
comparison = estimate_fvf(dists, ON_TARGET)
display(comparison.round(2))

pooled = comparison.loc['all']
per_subject = comparison.drop(index='all')

print(f'{"estimator":<20}{"pooled":>9}{"/on-target":>12}{"subj range":>16}{"verdict":>12}')
for name, label in [('foveation_falloff', 'A falloff'), ('launch_distance', 'B launch'),
                    ('selection_hazard', 'C hazard')]:
    value = pooled[name]
    col = per_subject[name].dropna()
    spread = f'{col.min():.2f}-{col.max():.2f}' if len(col) else 'n/a'
    if np.isnan(value):
        verdict = 'censored'
    elif not (ON_TARGET < value < 10):
        verdict = 'implausible'
    else:
        verdict = 'OK'
    ratio = 'n/a' if np.isnan(value) else f'{value / ON_TARGET:.1f}x'
    shown = 'NaN' if np.isnan(value) else f'{value:.2f}'
    print(f'{label:<20}{shown:>9}{ratio:>12}{spread:>16}{verdict:>12}')

print(f'\non-target threshold: {ON_TARGET:.2f} DVA (independently derived)')
print(f'array extent: ~34 x 19 DVA')

### Visualize

Left: A's foveation curve, which barely descends. Right: C's selection hazard, which falls by ~40x and has
a well-defined half-point. Row 2 shows each subject separately, so heterogeneity is visible before a pooled
number is adopted.

In [ ]:
fig = make_subplots(rows=2, cols=2, shared_xaxes=True,
                    column_titles=['A - foveation rate (saturated)', 'C - selection hazard (usable)'])

fig.add_trace(row=1, col=1, trace=go.Scatter(
    x=curve_a['centre'], y=curve_a['rate'], mode='lines+markers', name='P(foveated)',
    line=dict(color=cnfg.get_discrete_color('all'), width=3)))
fig.add_trace(row=1, col=2, trace=go.Scatter(
    x=curve_c['centre'], y=curve_c['rate'], mode='lines+markers', name='P(selected)',
    line=dict(color=cnfg.get_discrete_color('all'), width=3)))
fig.add_vline(x=hazard_pooled, row=1, col=2, line=dict(dash='dash', width=2))
fig.add_vline(x=ON_TARGET, row=1, col=2, line=dict(dash='dot', width=2, color='grey'))

for subj in sorted(opportunities['subject'].unique()):
    colour = cnfg.get_discrete_color(int(subj), loop=True)
    subset = fixations[fixations['subject'] == subj]
    sub_a = estimate_by_foveation_falloff(subset, ON_TARGET)[2]
    sub_c = estimate_by_selection_hazard(subset, ON_TARGET)[2]
    fig.add_trace(row=2, col=1, trace=go.Scatter(
        x=sub_a['centre'], y=sub_a['rate'], mode='lines', name=f'S{subj}',
        line=dict(color=colour, width=1.5), opacity=0.7, showlegend=False))
    fig.add_trace(row=2, col=2, trace=go.Scatter(
        x=sub_c['centre'], y=sub_c['rate'], mode='lines', name=f'S{subj}',
        line=dict(color=colour, width=1.5), opacity=0.7, showlegend=False))

fig.update_xaxes(title=dict(text='Distance from Fixation (DVA)', font=cnfg.AXIS_LABEL_FONT),
                 tickfont=cnfg.AXIS_TICK_FONT, gridcolor=cnfg.GRID_LINE_COLOR, row=2)
fig.update_yaxes(tickfont=cnfg.AXIS_TICK_FONT, gridcolor=cnfg.GRID_LINE_COLOR)
fig.update_layout(width=1300, height=700, paper_bgcolor='rgba(0, 0, 0, 0)',
                  title=dict(text='Functional Visual Field: why A saturates and C does not',
                             font=cnfg.TITLE_FONT))
fig.show()

### Landing the value

Nothing consumes the FVF yet - the d' denominator is a separate open task (`CODE_REVIEW.md` T1) and the
coverage/mapping work is deferred. **No constant is transcribed into `config.py` or `funnel_config.py`**: a
hand-copied per-subject dict would go stale the moment the pipeline is re-run. Callers should call
`estimate_fvf()` and pass a radius explicitly; this notebook justifies the value they choose.

When that consumer is written, use **`selection_hazard`** - and prefer the per-subject value over the
pooled one, since the whole purpose of the FVF denominator is to condition on what *that subject* could
plausibly have inspected.

### Threshold sensitivity: is the FVF a function of `ON_TARGET_THRESHOLD_DVA`?

Moved to its own notebook: `analysis/fvf/threshold_sweep.ipynb`. That constant enters all three
estimators in more than one place - it decides which fixations count as on-target, and therefore which
targets count as foveated, where each target's risk set ends, and which approaches count as peripheral -
so the estimates here are not obviously independent of it.